# Cross-product diagnostics — FS excess and batch effects

Reads `08_accumulate.ipynb`'s `*_merged.full.tsv` output (one file per
`(phenotype, transform, covariate_set)` combo) and diagnoses the source of
**elevated cross-products in the full-sib range (a ≈ 0.4–0.6) relative to
unrelated pairs** — i.e. positive b² (shared environment), PO-pair
contamination in the FS bin, or measurement batch effects. Run
`08_accumulate.ipynb` first; this notebook only reads its output.

## What can cause "high in sibs, low elsewhere"

Under a pure additive genetic model the expected cross-product E[y_i y_j] ≈ h²
a_ij at every relatedness level: the curve from unrelated through related pairs
should be one straight line through the origin. Systematic elevation at
a ≈ 0.4–0.6 with a lower slope among unrelated pairs indicates **one or more**
of the following:

1. **Genuine shared environment (b²):** siblings who grew up in the same
   household are more phenotypically similar than genetics alone predicts.
   Well-documented for social/behavioural traits (PSS, loneliness, AUDIT-C),
   SES-linked anthropometrics (BMI, WHR), and metabolic labs in large biobanks
   (UKBB, Kemper et al. 2021). Kemper et al. explicitly fit b² for all
   phenotypes and report nonzero estimates for several; UKBB sib-based analyses
   (Zhu et al. 2018, Border et al. 2022) find similar.

2. **PO-pair contamination (most likely pipeline cause here):** parent-offspring
   pairs (a_ij ≈ 0.5) land in the same bin as full sibs. The current pipeline
   (`grm_shard_tool`, `08_accumulate.ipynb`) does NOT remove PO pairs before
   accumulation — `07_relatedness_screen.ipynb` identifies them, but its
   `po_pairs_exclude.tsv` output is not yet wired into the accumulate step.
   PO pairs can have a different covariance structure (age gap → PCs partially
   decorrelate; no shared-childhood effect for adult-diagnosed phenotypes),
   inflating or deflating the FS-bin estimate depending on the phenotype.
   This is the primary motivation for the `grm_class_tool` being developed
   on this branch.

3. **Measurement batch effects:** siblings are measured at more similar times
   and sites than unrelated individuals. If lab calibration or collection
   protocol varies by site/date, siblings show spurious covariance. Distinguished
   from genuine b² by: (a) it would affect *all* phenotypes uniformly rather
   than clustering on social/household ones; (b) the Unrel.intercept diagnostic
   would be non-zero (correlated error among unrelated pairs from the same
   batch); (c) adding zip3/SES covariates should *not* reduce it (batch is
   orthogonal to neighbourhood SES).

4. **Distributional artefacts:** zero-inflation (cigarettes_per_day), bounded
   discrete scores (AUDIT-C, PSS), or residual skewness after INT can produce
   non-linear cross-product curves. Expect this to disappear on the invnorm
   scale for distributional artefacts.

## Sections

- **A. Cross-product scatter plots** — visual overview of the full curve per phenotype
- **B. FS excess quantified** — b²_FS and excess_FS across all combos
- **C. PO-pair count check** — how many PO pairs land in the FS bin
- **D. Step pattern** — b² in HS range (0.2–0.4) vs. FS range; distinguishes
  shared environment (step present in HS) from PO contamination (HS not affected)
- **E. Covariate-set absorption** — does b²_FS drop when adding zip3/SES?
- **F. Unrel.intercept** — batch-effect diagnostic among unrelated pairs
- **G. Raw vs. invnorm comparison** — does excess survive the skewness transform?

## Setup

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Must match 08_accumulate.ipynb's RUN_DIR -- points at the same WORK_DIR
RUN_DIR = "1kg_eur"

WORK_DIR = os.path.expanduser(f"~/grm_pheno_cov_{RUN_DIR}")

# Every row of a merged file is one (class, bin), so every bin selection has to
# name a class or it silently returns the same bin once per class plus once as
# `pooled`. `pooled` is merge's synthesis of all real classes and reproduces
# what an unclassed run gave, so it is the right default for diagnostics meant
# to be comparable with earlier runs. Section C below deliberately compares
# classes against each other instead.
DIAG_CLASS = "pooled"

PLOTS_DIR = os.path.join(WORK_DIR, "diagnostics_plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "font.size": 9})

print("WORK_DIR:", WORK_DIR)
print("DIAG_CLASS:", DIAG_CLASS)

## Discover merged combos

In [ ]:
merged_files = sorted(glob.glob(f"{WORK_DIR}/*_merged.full.tsv"))
assert merged_files, f"No *_merged.full.tsv found under {WORK_DIR} -- run 08_accumulate.ipynb first"

combos = []
for path in merged_files:
    stem = os.path.basename(path)[: -len("_merged.full.tsv")]
    parts = stem.split("__", 2)
    if len(parts) != 3:
        print(f"  skipping unparseable filename: {path}")
        continue
    phenotype_name, transform, covariate_set = parts
    combos.append(dict(phenotype_name=phenotype_name, transform=transform,
                       covariate_set=covariate_set, path=path))

phenotypes = sorted({c["phenotype_name"] for c in combos})
transforms = sorted({c["transform"] for c in combos})
covariate_sets = sorted({c["covariate_set"] for c in combos})

print(f"{len(combos)} combos  |  {len(phenotypes)} phenotypes  |  "
      f"{len(transforms)} transforms  |  {len(covariate_sets)} covariate sets")
print("phenotypes:", phenotypes)
print("covariate sets:", covariate_sets)

## Helper functions

In [ ]:
def usable(df, lo=-np.inf, hi=np.inf, cls=None):
    """Bins in [lo, hi) for ONE pair class, with pairs and a finite JK SE.

    The class filter is not optional: a merged file holds one row per
    (class, bin), so without it each bin is returned once per class plus once
    as `pooled` -- mixing classes and double-counting every pair.
    """
    cls = DIAG_CLASS if cls is None else cls
    return df[
        (df["class"] == cls)
        & (df["bin_midpoint"] >= lo) & (df["bin_midpoint"] < hi)
        & (df["full_n"] > 0)
        & df["jk_se"].notna() & (df["jk_se"] > 0)
    ]


def wls(a, y, w, intercept=False):
    """Weighted least squares. Returns (beta, se) or (None, None) if underdetermined."""
    X = np.column_stack([a, np.ones_like(a)]) if intercept else a[:, None]
    if len(y) <= X.shape[1]:
        return None, None
    XtW = X.T * w
    XtWX = XtW @ X
    if np.linalg.matrix_rank(XtWX) < X.shape[1]:
        return None, None
    beta = np.linalg.solve(XtWX, XtW @ y)
    dof = len(y) - X.shape[1]
    if dof <= 0:
        return beta, np.full_like(beta, np.nan)
    sigma2 = np.sum(w * (y - X @ beta) ** 2) / dof
    return beta, np.sqrt(np.diag(sigma2 * np.linalg.inv(XtWX)))


def diagnostics(df, cls=None):
    """Scalar summaries for one merged.full.tsv DataFrame, for one pair class."""
    def _sub(lo, hi):
        s = usable(df, lo, hi, cls)
        if len(s) == 0:
            return np.array([]), np.array([]), np.array([])
        return s["bin_midpoint"].to_numpy(), s["full_mean"].to_numpy(), 1.0 / s["jk_se"].to_numpy() ** 2

    # h2_Unrel: no-intercept slope on unrelated bins (a < 0.02)
    au, yu, wu = _sub(-np.inf, 0.02)
    beta_u, _ = wls(au, yu, wu, intercept=False)
    h2_unrel = float(beta_u[0]) if beta_u is not None else np.nan

    # Unrel.intercept: intercept from [slope, intercept] fit on unrelated bins
    beta_ui, se_ui = wls(au, yu, wu, intercept=True)
    unrel_int = float(beta_ui[1]) if beta_ui is not None else np.nan
    unrel_int_se = float(se_ui[1]) if se_ui is not None else np.nan

    # b2_FS: 2 * intercept of [slope, intercept] fit on FS bins (0.4 <= a < 0.6)
    af, yf, wf = _sub(0.4, 0.6)
    beta_fs, se_fs = wls(af, yf, wf, intercept=True)
    b2_fs = 2.0 * float(beta_fs[1]) if beta_fs is not None else np.nan
    b2_fs_se = 2.0 * float(se_fs[1]) if se_fs is not None else np.nan

    # excess_FS: pair-weighted mean cross-product in FS range minus additive prediction
    if np.isfinite(h2_unrel) and len(af) > 0:
        sf = usable(df, 0.4, 0.6, cls)
        n = sf["full_n"].to_numpy()
        excess_fs = float(np.average(yf, weights=n) - h2_unrel * np.average(af, weights=n))
    else:
        excess_fs = np.nan

    # b2_HS: same intercept model on half-sib range (0.2 <= a < 0.4)
    ah, yh, wh = _sub(0.2, 0.4)
    beta_hs, se_hs = wls(ah, yh, wh, intercept=True)
    b2_hs = 2.0 * float(beta_hs[1]) if beta_hs is not None else np.nan
    b2_hs_se = 2.0 * float(se_hs[1]) if se_hs is not None else np.nan

    return dict(
        h2_unrel=h2_unrel,
        unrel_int=unrel_int, unrel_int_se=unrel_int_se,
        b2_fs=b2_fs, b2_fs_se=b2_fs_se,
        b2_hs=b2_hs, b2_hs_se=b2_hs_se,
        excess_fs=excess_fs,
        n_unrel_bins=len(au), n_fs_bins=len(af), n_hs_bins=len(ah),
    )


print("helpers defined")

## Load all combos

In [ ]:
dfs = {}   # (phenotype_name, transform, covariate_set) -> DataFrame
for c in combos:
    key = (c["phenotype_name"], c["transform"], c["covariate_set"])
    df = pd.read_csv(c["path"], sep="\t")
    if "class" not in df.columns:
        # pre-classification output: one implicit class, which is what `pooled` means
        df["class"] = "pooled"
    dfs[key] = df

print(f"loaded {len(dfs)} DataFrames")
sample = next(iter(dfs.values()))
AVAILABLE_CLASSES = sorted(sample["class"].unique())
print("columns:", list(sample.columns))
print("classes:", AVAILABLE_CLASSES)
assert DIAG_CLASS in AVAILABLE_CLASSES, (
    f"DIAG_CLASS={DIAG_CLASS!r} is not in {AVAILABLE_CLASSES}"
)
sample.head(3)

## A. Cross-product scatter plots

One row per phenotype, one column per covariate set (if multiple). Each panel
shows the binned cross-product E[y_i y_j] vs. GRM relatedness a_ij, with
±1 JK-SE error bars. The red line is the h²_Unrel no-intercept WLS fit over
unrelated bins (a < 0.02), extended across the full range — deviations above
this line in the FS range (0.4–0.6, shaded) are the excess this notebook
targets.

**Expected shape under pure additive genetics:** a straight line through the
origin from unrelated through full-sib pairs. Any systematic elevation at
a ≈ 0.5 relative to this line is the "high in sibs" pattern.

In [ ]:
# Use one fixed transform for the overview (prefer invnorm; fall back to raw)
OVERVIEW_TRANSFORM = "invnorm" if "invnorm" in transforms else transforms[0]

cs_colors = plt.cm.viridis(np.linspace(0, 1, len(covariate_sets)))
cs_color_map = dict(zip(covariate_sets, cs_colors))

ncols = len(covariate_sets)
nrows = len(phenotypes)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.5 * nrows),
                          squeeze=False, constrained_layout=True)

for row_i, pheno in enumerate(phenotypes):
    for col_j, cs in enumerate(covariate_sets):
        ax = axes[row_i, col_j]
        key = (pheno, OVERVIEW_TRANSFORM, cs)
        if key not in dfs:
            ax.set_visible(False)
            continue

        df = dfs[key]
        sub = usable(df)
        a = sub["bin_midpoint"].to_numpy()
        y = sub["full_mean"].to_numpy()
        se = sub["jk_se"].to_numpy()

        ax.errorbar(a, y, yerr=se, fmt=".", ms=3, lw=0.5,
                    color=cs_color_map[cs], alpha=0.7, zorder=3)

        # h2_Unrel fit line extended to 0.7
        u_sub = usable(df, -np.inf, 0.02)
        au = u_sub["bin_midpoint"].to_numpy()
        beta_u, _ = wls(au, u_sub["full_mean"].to_numpy(),
                        1.0 / u_sub["jk_se"].to_numpy() ** 2, intercept=False)
        if beta_u is not None:
            x_line = np.linspace(0, 0.7, 200)
            ax.plot(x_line, beta_u[0] * x_line, color="firebrick",
                    lw=1.2, zorder=4, label=f"h²_Unrel={beta_u[0]:.3f}")

        # shade FS range
        ax.axvspan(0.4, 0.6, color="gold", alpha=0.12, zorder=0)
        ax.axvline(0.02, color="grey", lw=0.6, ls=":", zorder=1)
        ax.axvline(0.05, color="grey", lw=0.6, ls=":", zorder=1)
        ax.axhline(0, color="grey", lw=0.5)

        ax.set_title(f"{pheno}\n{cs}", fontsize=7.5)
        ax.set_xlabel("a_ij", fontsize=7)
        ax.set_ylabel("E[y_i y_j]", fontsize=7)
        ax.legend(fontsize=6, loc="upper left")

fig.suptitle(f"Cross-product bins — {RUN_DIR}, {OVERVIEW_TRANSFORM} transform\n"
             f"(gold shading = FS range 0.4–0.6; red = h²_Unrel fit)",
             fontsize=10, y=1.001)
fig.savefig(f"{PLOTS_DIR}/scatter_overview_{OVERVIEW_TRANSFORM}.png", dpi=150,
            bbox_inches="tight")
plt.show()
print(f"Saved to {PLOTS_DIR}/scatter_overview_{OVERVIEW_TRANSFORM}.png")

## B. FS excess quantified

- **b²_FS** = 2 × intercept of the FS-range regression [slope, intercept] × a_ij.
  Under pure additive genetics b²_FS = 0. A positive value means the FS-bin
  mean is elevated above what the unrelated-pair slope predicts.
- **excess_FS** = (pair-weighted mean cross-product in FS range) − h²_Unrel ×
  (pair-weighted mean a_ij in FS range). Non-parametric version of b²_FS;
  no regression model assumed.

Forest plot: each row is a (phenotype, covariate_set) pair, faceted by
transform. Error bars are analytic WLS SEs.

In [ ]:
diag_rows = []
for c in combos:
    key = (c["phenotype_name"], c["transform"], c["covariate_set"])
    d = diagnostics(dfs[key])
    diag_rows.append({**c, **d})

diag = pd.DataFrame(diag_rows).drop(columns=["path"])

# order phenotypes by descending mean b2_fs (across all transforms/covsets)
pheno_order = (
    diag.groupby("phenotype_name")["b2_fs"].mean()
    .sort_values(ascending=False).index.tolist()
)
diag["phenotype_name"] = pd.Categorical(diag["phenotype_name"], categories=pheno_order, ordered=True)
diag = diag.sort_values(["phenotype_name", "covariate_set", "transform"])

print(f"Diagnostics computed for {len(diag)} combos")
diag[["phenotype_name", "transform", "covariate_set",
      "h2_unrel", "b2_fs", "b2_fs_se", "excess_fs", "b2_hs"]].round(4)

In [ ]:
def forest_b2(diag_subset, title, fname):
    """Forest plot of b2_fs by phenotype × covariate_set for one transform."""
    n_pheno = len(diag_subset["phenotype_name"].cat.categories)
    cs_list = sorted(diag_subset["covariate_set"].unique())
    offsets = np.linspace(-0.2, 0.2, len(cs_list))

    fig, ax = plt.subplots(figsize=(7, max(4, 0.4 * n_pheno + 2)))
    for j, cs in enumerate(cs_list):
        sub = diag_subset[diag_subset["covariate_set"] == cs].sort_values("phenotype_name")
        y_pos = [pheno_order.index(p) + offsets[j] for p in sub["phenotype_name"]]
        ax.errorbar(sub["b2_fs"], y_pos, xerr=sub["b2_fs_se"].clip(lower=0),
                    fmt="o", ms=4, lw=1.2, label=cs,
                    color=cs_color_map.get(cs, "black"))

    ax.axvline(0, color="grey", lw=0.8, ls="--")
    ax.set_yticks(range(n_pheno))
    ax.set_yticklabels(pheno_order, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel("b²_FS (± analytic SE)")
    ax.set_title(title, fontsize=9)
    ax.legend(title="covariate set", fontsize=7, loc="lower right")
    ax.axvspan(0, ax.get_xlim()[1], color="tomato", alpha=0.05)  # positive = excess
    plt.tight_layout()
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


for tr in transforms:
    sub = diag[diag["transform"] == tr]
    if sub.empty:
        continue
    forest_b2(
        sub,
        title=f"b²_FS by phenotype — {RUN_DIR}, {tr} transform",
        fname=f"{PLOTS_DIR}/b2_fs_forest_{tr}.png",
    )
    print(f"Saved b2_fs_forest_{tr}.png")

## C. PO contamination of the FS bin — measured, not estimated

Parent-offspring pairs sit at `a_ij ~= 0.5` and share a bin with full sibs. This
section used to approximate their contribution by counting rows in
`po_pairs_exclude.tsv` and dividing by the FS-bin pair count from an unrelated
combo — which mixed two different denominators.

With `--pair-classes` the answer is exact and comes from the same file: the
`PO` and `FS` classes hold the same bins over 0.4–0.6, with their own pair
counts, means and jackknife SEs. So the PO fraction is a ratio of pair counts
within one combo, and — more usefully — the **excess and `b2` can be recomputed
per class**, which says directly how much of the FS excess was PO contamination
rather than leaving it a plausible story.

`pooled` is the pre-classification number. If `b2` on `FS` differs materially
from `b2` on `pooled`, the separation mattered.

In [ ]:
if not {"PO", "FS"} <= set(AVAILABLE_CLASSES):
    print(f"classes available: {AVAILABLE_CLASSES}")
    print("No PO/FS split in this output — re-run 08_accumulate.ipynb with")
    print("--pair-classes (deg1_classified.tsv from 07_relatedness_screen.ipynb).")
else:
    rows = []
    for (phe, tr, cs), df in dfs.items():
        counts = {c: int(usable(df, 0.4, 0.6, c)["full_n"].sum())
                  for c in AVAILABLE_CLASSES}
        n_pooled = counts.get("pooled", 0)
        row = dict(phenotype=phe, transform=tr, covariate_set=cs,
                   n_PO=counts.get("PO", 0), n_FS=counts.get("FS", 0),
                   n_pooled=n_pooled,
                   pct_PO=(100 * counts.get("PO", 0) / n_pooled) if n_pooled else np.nan)
        for c in ("FS", "PO", "pooled"):
            d = diagnostics(df, c)
            row[f"b2_{c}"] = d["b2_fs"]
            row[f"excess_{c}"] = d["excess_fs"]
        rows.append(row)

    fs_po = pd.DataFrame(rows)
    pct = fs_po["pct_PO"].dropna()
    print(f"PO share of pairs in the 0.4-0.6 band: "
          f"{pct.min():.1f}% - {pct.max():.1f}% (median {pct.median():.1f}%)")
    print(f"  {fs_po['n_PO'].iloc[0]:,} PO vs {fs_po['n_FS'].iloc[0]:,} FS pairs "
          f"of {fs_po['n_pooled'].iloc[0]:,} total in that band\n")

    if pct.median() > 5:
        print("PO pairs are >5% of the band: pooling them into the FS bin biases")
        print("b2_FS and the FS excess. Compare the columns below to see by how much.")
    else:
        print("PO pairs are <5% of the band, so PO contamination alone cannot")
        print("explain a large FS excess — look to shared environment or dominance.")

    shift = (fs_po["b2_FS"] - fs_po["b2_pooled"]).abs()
    print(f"\n|b2 on FS - b2 on pooled|: median {shift.median():.4f}, "
          f"max {shift.max():.4f}")
    print("  (how much the PO separation moved the shared-environment estimate)")
    fs_po.round(4).head(20)

## D. Step pattern — FS vs. HS range

Under genuine shared environment (b²), the intercept should appear at ALL
close-relative ranges: full sibs AND half-sibs/2nd-degree relatives share
some household/neighbourhood environment (though more of it for FS than HS).

Under PO contamination, PO pairs concentrate in the FS bin (a ≈ 0.5) but NOT
in the HS range (a ≈ 0.2–0.4). So:
- **b²_FS >> b²_HS ≈ 0** → PO contamination more likely
- **b²_FS > b²_HS > 0** → genuine shared environment more likely (step pattern)

The scatter below compares b²_FS vs. b²_HS per phenotype.

In [ ]:
for tr in transforms:
    sub = diag[(diag["transform"] == tr) & (diag["covariate_set"] == covariate_sets[0])].copy()
    if sub.empty:
        continue

    fig, ax = plt.subplots(figsize=(6, 5))
    sc = ax.scatter(sub["b2_hs"], sub["b2_fs"], c="steelblue", s=60, zorder=3)
    for _, row in sub.iterrows():
        ax.annotate(row["phenotype_name"], (row["b2_hs"], row["b2_fs"]),
                    fontsize=6.5, ha="left", va="bottom",
                    xytext=(3, 2), textcoords="offset points")

    # b2_FS == b2_HS line (shared environment with equal effect in both ranges)
    # np.nanabs does not exist; Series.abs() already skips NaN, and nanmax
    # guards the all-NaN case.
    lim = np.nanmax([sub["b2_fs"].abs().max(), sub["b2_hs"].abs().max()]) * 1.1
    if not np.isfinite(lim) or lim == 0:
        lim = 1.0
    ax.plot([-lim, lim], [-lim, lim], color="grey", lw=0.8, ls="--", label="b²_FS = b²_HS")
    ax.axhline(0, color="grey", lw=0.5)
    ax.axvline(0, color="grey", lw=0.5)

    ax.set_xlabel("b²_HS (half-sib / 2nd-degree, 0.2–0.4)")
    ax.set_ylabel("b²_FS (full-sib range, 0.4–0.6)")
    ax.set_title(f"Step pattern: b²_FS vs. b²_HS — {RUN_DIR}, {tr}\n"
                 f"({covariate_sets[0]}; points above diagonal = FS excess > HS excess)",
                 fontsize=8.5)
    ax.legend(fontsize=7)
    plt.tight_layout()
    fname = f"{PLOTS_DIR}/step_pattern_{tr}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved {fname}")

## E. Covariate-set absorption test

If the FS excess is driven by **geographic/SES confounding** (siblings living
in the same zip code, which predicts phenotype), b²_FS should drop substantially
when zip3 and SES variables are added to the covariate set.

If the excess is driven by **PO contamination** or **genetic dominance**, adding
covariates should have minimal effect.

If the excess is driven by **measurement batch effects** (siblings measured
at the same site/date), adding covariates likewise won't help — you'd need to
add measurement site/date as explicit covariates (currently not pulled).

Plot: b²_FS across the covariate staircase for each phenotype.

In [ ]:
cs_order = [cs for cs in ["base", "base_pcs", "base_pcs_zip3", "base_pcs_zip3_ses"]
            if cs in covariate_sets]
cs_order = cs_order if cs_order else covariate_sets

for tr in transforms:
    sub = diag[diag["transform"] == tr].copy()
    if sub.empty:
        continue
    sub["covariate_set"] = pd.Categorical(sub["covariate_set"], categories=cs_order, ordered=True)

    n_pheno = len(phenotypes)
    ncols = 4
    nrows_p = (n_pheno + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows_p, ncols, figsize=(3.5 * ncols, 2.8 * nrows_p),
                              squeeze=False, constrained_layout=True)

    for i, pheno in enumerate(pheno_order):
        ax = axes[i // ncols][i % ncols]
        p_sub = sub[sub["phenotype_name"] == pheno].sort_values("covariate_set")
        if p_sub.empty:
            ax.set_visible(False)
            continue
        ax.errorbar(range(len(p_sub)), p_sub["b2_fs"], yerr=p_sub["b2_fs_se"].clip(lower=0),
                    fmt="o-", ms=5, lw=1.2, color="steelblue")
        ax.axhline(0, color="grey", lw=0.5, ls="--")
        ax.set_xticks(range(len(p_sub)))
        ax.set_xticklabels([cs.replace("base_", "") for cs in p_sub["covariate_set"]],
                           rotation=35, ha="right", fontsize=6.5)
        ax.set_title(pheno, fontsize=7.5)
        ax.set_ylabel("b²_FS", fontsize=7)

    for i in range(len(pheno_order), nrows_p * ncols):
        axes[i // ncols][i % ncols].set_visible(False)

    fig.suptitle(f"b²_FS across covariate staircase — {RUN_DIR}, {tr}\n"
                 f"(drop from base→base_pcs_zip3_ses = geographic/SES absorption)",
                 fontsize=9)
    fname = f"{PLOTS_DIR}/covset_absorption_{tr}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved {fname}")

## F. Unrel.intercept — batch-effect diagnostic

Among truly unrelated pairs (a < 0.02), E[y_i y_j] ≈ 0 once phenotypes are
standardized to mean 0. A systematic non-zero **intercept** in this range
indicates that unrelated individuals from the *same batch or site* are
positively correlated — a signal of **measurement batch effects** rather
than genetic or environmental relatedness (Kemper et al. 2021, Methods;
also see the `h2_UnrelInt`/`Unrel.intercept` diagnostics in
`src/he_estimators.py`).

**Distinguishes batch effects from shared environment:** genuine shared
environment only elevates cross-products for *relatives* (sibs); batch
effects inflate cross-products for *co-batch* individuals, who are often
unrelated. So a large Unrel.intercept + large b²_FS = suspect batch;
Unrel.intercept ≈ 0 + large b²_FS = genuine shared environment or PO
contamination.

In [ ]:
for tr in transforms:
    sub = diag[diag["transform"] == tr].sort_values("phenotype_name")
    if sub.empty:
        continue

    cs_list = sorted(sub["covariate_set"].unique())
    offsets = np.linspace(-0.2, 0.2, len(cs_list))
    n_pheno = len(pheno_order)

    fig, ax = plt.subplots(figsize=(7, max(4, 0.4 * n_pheno + 2)))
    for j, cs in enumerate(cs_list):
        s = sub[sub["covariate_set"] == cs].sort_values("phenotype_name")
        y_pos = [pheno_order.index(p) + offsets[j] for p in s["phenotype_name"]]
        ax.errorbar(s["unrel_int"], y_pos,
                    xerr=s["unrel_int_se"].clip(lower=0),
                    fmt="o", ms=4, lw=1.2, label=cs,
                    color=cs_color_map.get(cs, "black"))

    ax.axvline(0, color="grey", lw=0.8, ls="--")
    ax.set_yticks(range(n_pheno))
    ax.set_yticklabels(pheno_order, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel("Unrel.intercept (± analytic SE)")
    ax.set_title(
        f"Unrel.intercept diagnostic — {RUN_DIR}, {tr}\n"
        "(non-zero = residual batch effect or population structure among unrelated pairs)",
        fontsize=8.5
    )
    ax.legend(title="covariate set", fontsize=7, loc="lower right")
    plt.tight_layout()
    fname = f"{PLOTS_DIR}/unrel_intercept_{tr}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved {fname}")

## G. Raw vs. invnorm comparison

If the FS excess is a **distributional artefact** (zero-inflation in
`cigarettes_per_day`, bounded integer scores in AUDIT-C/PSS/loneliness,
or residual skewness), it should be substantially reduced or eliminated
on the invnorm scale. If it survives the rank-INT transform, the excess
reflects genuine cross-product structure, not a distributional quirk.

Scatter plot: b²_FS on raw scale vs. b²_FS on invnorm scale, per phenotype.
Points on the diagonal mean the excess is the same on both scales; points
above mean the invnorm excess is larger (unusual); points below mean
invnorm reduced it (suggests some distributional contribution).

In [ ]:
if "raw" in transforms and "invnorm" in transforms:
    # Use the richest common covariate set for this comparison
    common_cs = cs_order[-1] if cs_order else covariate_sets[-1]

    raw_b2 = (
        diag[(diag["transform"] == "raw") & (diag["covariate_set"] == common_cs)]
        .set_index("phenotype_name")[["b2_fs", "b2_fs_se"]]
        .rename(columns={"b2_fs": "b2_raw", "b2_fs_se": "se_raw"})
    )
    inv_b2 = (
        diag[(diag["transform"] == "invnorm") & (diag["covariate_set"] == common_cs)]
        .set_index("phenotype_name")[["b2_fs", "b2_fs_se"]]
        .rename(columns={"b2_fs": "b2_inv", "b2_fs_se": "se_inv"})
    )
    both = raw_b2.join(inv_b2, how="inner")

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.errorbar(both["b2_raw"], both["b2_inv"],
                xerr=both["se_raw"].clip(lower=0),
                yerr=both["se_inv"].clip(lower=0),
                fmt=".", ms=0, lw=0.6, color="grey", alpha=0.6, zorder=2)
    ax.scatter(both["b2_raw"], both["b2_inv"], s=50, color="steelblue", zorder=3)

    for pheno, row in both.iterrows():
        ax.annotate(pheno, (row["b2_raw"], row["b2_inv"]),
                    fontsize=6, ha="left", va="bottom",
                    xytext=(3, 2), textcoords="offset points")

    lim = max(np.nanmax(np.abs(both[["b2_raw", "b2_inv"]].values)), 0.01) * 1.15
    ax.plot([-lim, lim], [-lim, lim], color="firebrick", lw=0.8, ls="--",
            label="raw = invnorm")
    ax.axhline(0, color="grey", lw=0.4)
    ax.axvline(0, color="grey", lw=0.4)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel("b²_FS (raw scale)")
    ax.set_ylabel("b²_FS (invnorm scale)")
    ax.set_title(f"Raw vs. invnorm b²_FS — {RUN_DIR}, {common_cs}\n"
                 "(below diagonal = invnorm reduced the excess; distributional contribution)",
                 fontsize=8.5)
    ax.legend(fontsize=7)
    plt.tight_layout()
    fname = f"{PLOTS_DIR}/raw_vs_invnorm_b2.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved {fname}")
else:
    print("Both 'raw' and 'invnorm' transforms required for this comparison — skipping.")

## Summary table

Consolidated view of all diagnostics, sorted by descending excess_FS on the
invnorm + richest covariate set. Use this to prioritise which phenotypes need
the closest attention.

**Reading guide:**
- `excess_fs` > 0.05 and `unrel_int` ≈ 0 → likely genuine shared environment
  or PO contamination (check the step-pattern plot: if b2_hs ≈ 0, lean PO)
- `excess_fs` > 0.05 and `unrel_int` > 0 → possible measurement batch effect;
  adding measurement site/date as a covariate is the fix
- `excess_fs` reduced by invnorm (see section G) → partial distributional artefact
- `excess_fs` reduced by richer covariate set (section E) → geographic/SES confounding

In [ ]:
# Focus on invnorm + richest covariate set for the summary
SUMMARY_TRANSFORM = "invnorm" if "invnorm" in transforms else transforms[0]
SUMMARY_CS = cs_order[-1] if cs_order else covariate_sets[-1]

summary = (
    diag[(diag["transform"] == SUMMARY_TRANSFORM) & (diag["covariate_set"] == SUMMARY_CS)]
    .sort_values("excess_fs", ascending=False)
    [["phenotype_name", "h2_unrel", "b2_fs", "b2_fs_se",
      "b2_hs", "b2_hs_se", "excess_fs", "unrel_int", "unrel_int_se",
      "n_unrel_bins", "n_fs_bins"]]
    .reset_index(drop=True)
)

print(f"Summary: {SUMMARY_TRANSFORM} transform, {SUMMARY_CS} covariate set")
print()

# flag phenotypes with large excess
FLAG_THRESH = 0.05   # excess_fs > this is flagged as notable
flagged = summary[summary["excess_fs"].abs() > FLAG_THRESH]["phenotype_name"].tolist()
if flagged:
    print(f"Phenotypes with |excess_FS| > {FLAG_THRESH}: {flagged}")
else:
    print(f"No phenotypes with |excess_FS| > {FLAG_THRESH} (threshold can be adjusted above)")
print()

pd.set_option("display.float_format", "{:.4f}".format)
summary

In [ ]:
out_tsv = os.path.join(WORK_DIR, f"crossproduct_diagnostics_{RUN_DIR}.tsv")
diag.round(6).to_csv(out_tsv, sep="\t", index=False)
print(f"Full diagnostics table written to {out_tsv}")
print(f"All plots saved to {PLOTS_DIR}/")